In [ ]:
%pip install pandas

In [ ]:
import pandas as pd

# Naïve Bayes

Основывается на формуле Байеса (Расчет условной вероятности),\
которая используется, когда известна P( B | A ), а нужно узнать P( A | B )
$$ P(A|B) = \frac{P(B | A) * P(A)}{P(B)}$$


Почему "Наивный" байес? При работе с данными мы производим допущение, что все признаки независимы друг от друга (Ну или, если быть более точным, предполагаем, что их зависимость принимает пренебрежимо малые значения)

## Работа алгоритма

### - Fit -

1) Строится частотная таблица / словарь (Каждому значению фактора сопостовляются условные вероятности его встретить при той или иной целевой переменной). После чего применяется сглаживание Лапласа:
   $$P(Значение_{фактора} \mid Значение_{целевой}) = \frac{count(Значение_{фактора}, Значение_{целевой}) + \alpha}{Общее\_количество\_наборов\_с\_такой\_целевой + \alpha \cdot V}$$
   $$P(Значения_{целевой}) = \frac{Количество объектов Значения_{целевой}}{Количество объектов в датасете}$$
   , где α - коэффициент сглаживания (Обычно единица),\
   а V - размер словаря (То есть количество различных значений, которые принимает фактор в датасете)

В конце логарифмируют посчитанные априорные вероятности для целевой и условные вероятности для значений признаков вероятности - делается это лишь для того, чтобы при последующем перемножении не вышло слишком маленькое число, которое float просто округлит в ноль
\
\
\
Обычно используют натуральный логарифм, но, в целом, разницы в основании особой нет, тк 
$$\log_b(x) = \frac{ln(x)}{ln(b)}$$
то есть при замене основания у нас просто произойдет домножение на $$ \frac{1}{ln(b)} $$ в следствие чего ответ модели не поменяется, тк домножение на константу неравенства не меняет его решения
\
\
\
После обучения сложностью ```O(n)``` выходит следующий результат:
- ```ln(P(y = 0))```
- ```ln(P(y = 1))```
- Для каждого признака строится таблица (В памяти часто хранится как словарь словарей словарей по типу ключ-признак к значению, у которого ключ-значение_фактора к значению, у которого ключ-значение_целевой к значению логарифма условной вероятности значения признака при известном значениии целевой)
Пример таблички для признака (Для каждого признака хранится такая таблица)

| Значение признака | Вероятность при значении целевой = 0 | Вероятность при значении целевой = 0 |
|:-:|:-:|:-:|
| $$Значение_1$$ | $$ln(P(Значение_1 \mid Целевая=0))$$|  $$ln(P(Значение_1 \mid Целевая=1))$$ |
| $$Значение_2$$ | $$ln(P(Значение_2 \mid Целевая=0))$$|  $$ln(P(Значение_2 \mid Целевая=1))$$ |
| ... | ... | ... |
| $$Значение_k$$ | $$ln(P(Значение_k \mid Целевая=0))$$|  $$ln(P(Значение_k \mid Целевая=1))$$ |

### - Inference -

Считаются условные вероятности:\
Для y = 1:
$$ P(y = 1 \mid X_{1 \dots n}) = \frac{P(y = 1) \cdot \prod_{i=1}^{n} P(x_i \mid y = 1)}{P(X_1, \dots, X_n)} $$
Для y = 1:
$$ P(y = 0 \mid X_{1 \dots n}) = \frac{P(y = 0) \cdot \prod_{i=1}^{n} P(x_i \mid y = 0)}{P(X_1, \dots, X_n)} $$

Теперь сравниваем
$$ P(y = 0 \mid X_{1 \dots n}) \ \lessgtr \ P(y = 1 \mid X_{1 \dots n}) $$
$$ \frac{P(y = 0) \cdot \prod_{i=1}^{n} P(x_i \mid y = 0)}{P(X_1, \dots, X_n)} \ \lessgtr \ \frac{P(y = 1) \cdot \prod_{i=1}^{n} P(x_i \mid y = 1)}{P(X_1, \dots, X_n)} $$
Очевидно, знаменатели равны и ненулевые (Тк иначе мы бы никогда не встретили исследуемый набор) => домножим на него
$$ P(y = 0) \cdot \prod_{i=1}^{n} P(x_i \mid y = 0)  \ \lessgtr \ P(y = 1) \cdot \prod_{i=1}^{n} P(x_i \mid y = 1) $$
Вспомним про округление float и что может произведение значений, меньших единицы, округлиться в ноль и прологорифмируем
$$ ln(P(y = 0) \cdot \prod_{i=1}^{n} P(x_i \mid y = 0)) \ \lessgtr \ ln(P(y = 1) \cdot \prod_{i=1}^{n} P(x_i \mid y = 1)) $$
Дальше вспомним, что логарифм произведения равняется сумме логарифмов множителей
$$ ln(P(y = 0)) + \sum_{i=1}^{n} ln(P(x_i \mid y = 0)) \ \lessgtr \ ln(P(y = 1)) + \sum_{i=1}^{n} ln(P(x_i \mid y = 1)) $$

Все необходимые данные для решения неравенства были посчитаны на этапе обучения, считаем ```y=1``` и ```y=0``` и даем ответ, что больше - это и будет ответом нашей модели